# ML Pipeline — Late Delivery Prediction

**Goal:** Train a classifier that predicts `late_delivery` for an order given features known at order/shipment creation time. The trained model is saved to `jobs/model.pkl` and used by `jobs/run_inference.py` to populate `order_predictions`.

In [5]:
import sqlite3
import pathlib
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, roc_auc_score

DB_PATH = pathlib.Path('shop.db')
JOBS_DIR = pathlib.Path('jobs')
JOBS_DIR.mkdir(exist_ok=True)
MODEL_PATH = JOBS_DIR / 'model.pkl'

print('DB exists:', DB_PATH.exists())

DB exists: True


## 2. Load Data

In [6]:
con = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        s.order_id,
        s.carrier,
        s.shipping_method,
        s.distance_band,
        s.promised_days,
        s.late_delivery,
        o.order_total,
        o.payment_method,
        o.device_type,
        o.promo_used,
        o.risk_score,
        c.customer_segment,
        c.loyalty_tier
    FROM shipments s
    JOIN orders o ON o.order_id = s.order_id
    JOIN customers c ON c.customer_id = o.customer_id
""", con)

con.close()

print(df.shape)
df.head()

(5000, 13)


,order_id,carrier,shipping_method,distance_band,promised_days,late_delivery,order_total,payment_method,device_type,promo_used,risk_score,customer_segment,loyalty_tier
0,1,UPS,expedited,regional,3,1,724.69,card,mobile,0,38.3,standard,silver
1,2,FedEx,expedited,local,2,1,944.27,card,desktop,1,94.9,standard,silver
2,3,FedEx,expedited,national,4,1,850.85,card,mobile,0,53.8,standard,silver
3,4,UPS,standard,regional,6,0,156.47,bank,mobile,1,4.2,standard,silver
4,5,USPS,standard,regional,6,1,25.46,card,mobile,0,4.9,standard,silver


## 3. Exploratory Data Analysis

In [7]:
print('--- Target distribution ---')
print(df['late_delivery'].value_counts(normalize=True).round(3))

print('\n--- Late rate by carrier ---')
print(df.groupby('carrier')['late_delivery'].mean().round(3).sort_values(ascending=False))

print('\n--- Late rate by distance_band ---')
print(df.groupby('distance_band')['late_delivery'].mean().round(3).sort_values(ascending=False))

print('\n--- Late rate by shipping_method ---')
print(df.groupby('shipping_method')['late_delivery'].mean().round(3).sort_values(ascending=False))

print('\n--- Nulls ---')
print(df.isnull().sum())

--- Target distribution ---
late_delivery
1    0.596
0    0.404
Name: proportion, dtype: float64

--- Late rate by carrier ---
carrier
USPS     0.729
UPS      0.609
FedEx    0.500
Name: late_delivery, dtype: float64

--- Late rate by distance_band ---
distance_band
regional    0.601
local       0.593
national    0.591
Name: late_delivery, dtype: float64

--- Late rate by shipping_method ---
shipping_method
expedited    0.601
standard     0.596
overnight    0.581
Name: late_delivery, dtype: float64

--- Nulls ---
order_id            0
carrier             0
shipping_method     0
distance_band       0
promised_days       0
late_delivery       0
order_total         0
payment_method      0
device_type         0
promo_used          0
risk_score          0
customer_segment    0
loyalty_tier        0
dtype: int64


## 4. Feature Engineering

In [8]:
# Fill nulls for optional customer fields
df['customer_segment'] = df['customer_segment'].fillna('unknown')
df['loyalty_tier'] = df['loyalty_tier'].fillna('none')

CATEGORICAL_FEATURES = [
    'carrier',
    'shipping_method',
    'distance_band',
    'payment_method',
    'device_type',
    'customer_segment',
    'loyalty_tier',
]

NUMERIC_FEATURES = [
    'promised_days',
    'order_total',
    'promo_used',
    'risk_score',
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = 'late_delivery'

X = df[FEATURES]
y = df[TARGET]

print('Features:', FEATURES)
print('X shape:', X.shape)

Features: ['carrier', 'shipping_method', 'distance_band', 'payment_method', 'device_type', 'customer_segment', 'loyalty_tier', 'promised_days', 'order_total', 'promo_used', 'risk_score']
X shape: (5000, 11)


## 5. Train / Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train)}  Test: {len(X_test)}')

Train: 4000  Test: 1000


## 6. Build & Train Pipeline

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CATEGORICAL_FEATURES),
    ('num', 'passthrough', NUMERIC_FEATURES),
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
print('Training complete.')

Training complete.


## 7. Evaluate

In [11]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['on-time', 'late']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

              precision    recall  f1-score   support

     on-time       0.65      0.34      0.44       404
        late       0.66      0.88      0.75       596

    accuracy                           0.66      1000
   macro avg       0.66      0.61      0.60      1000
weighted avg       0.66      0.66      0.63      1000

ROC-AUC: 0.6793


In [12]:
# Feature importances
importances = model.named_steps['classifier'].feature_importances_
feat_df = pd.Series(importances, index=FEATURES).sort_values(ascending=False)
print('--- Feature importances ---')
print(feat_df.round(4))

--- Feature importances ---
order_total         0.3106
risk_score          0.1861
carrier             0.1517
loyalty_tier        0.0907
customer_segment    0.0789
promised_days       0.0460
payment_method      0.0318
distance_band       0.0310
device_type         0.0308
promo_used          0.0228
shipping_method     0.0196
dtype: float64


## 8. Save Model

In [13]:
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

Model saved to jobs/model.pkl


## 9. Smoke Test Inference

Verify the saved model can score a sample row — same logic `run_inference.py` will use.

In [14]:
loaded_model = joblib.load(MODEL_PATH)

sample = X_test.iloc[:5].copy()
probs = loaded_model.predict_proba(sample)[:, 1]
preds = loaded_model.predict(sample)

sample['late_delivery_probability'] = probs.round(4)
sample['predicted_late_delivery'] = preds
sample['actual_late_delivery'] = y_test.iloc[:5].values

print(sample[['late_delivery_probability', 'predicted_late_delivery', 'actual_late_delivery']])

      late_delivery_probability  predicted_late_delivery  actual_late_delivery
4287                     0.7650                        1                     1
2694                     0.5129                        1                     1
766                      0.4266                        0                     1
4223                     0.4324                        0                     0
4193                     0.5651                        1                     1
